<a href="https://colab.research.google.com/github/anubhavtiwari-cyber/ai-ml-internship-maincrafts/blob/main-maincraft/AI_ML_Task2_Model_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 AI/ML Task 2 — Feature Engineering, Model Optimization & Performance Comparison
**Maincrafts Technology Internship**  
**Dataset:** California Housing Dataset  
**Models:** Linear Regression | Ridge Regression | Decision Tree Regressor

## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ Libraries imported successfully!')

✅ Libraries imported successfully!


## Step 2: Load the Dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('HousePrice')], axis=1)

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Dataset Info:')
df.info()
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

## Step 3: Exploratory Data Analysis (EDA) — Graphs

In [ ]:
# --- Graph 1: Distribution of Target Variable ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['HousePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of House Prices', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['HousePrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Boxplot of House Prices', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Median House Value ($100k)')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['HousePrice'])

plt.tight_layout()
plt.suptitle('Graph 1: Target Variable Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('graph1_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 1 saved!')

In [ ]:
# --- Graph 2: Feature Distributions ---
features = data.feature_names
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feature in enumerate(features):
    axes[i].hist(df[feature], bins=40, color=sns.color_palette('Set2')[i % 8], edgecolor='white')
    axes[i].set_title(f'{feature}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.suptitle('Graph 2: Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('graph2_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 2 saved!')

In [ ]:
# --- Graph 3: Correlation Heatmap ---
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Graph 3: Correlation Heatmap of All Features', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('graph3_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 3 saved!')

In [ ]:
# --- Graph 4: Top Feature Correlations with Target ---
target_corr = df.corr()['HousePrice'].drop('HousePrice').sort_values()

colors = ['#d73027' if v < 0 else '#1a9850' for v in target_corr.values]
plt.figure(figsize=(10, 6))
bars = plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Graph 4: Feature Correlation with House Price', fontsize=15, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')

for bar, val in zip(bars, target_corr.values):
    plt.text(val + 0.005 if val >= 0 else val - 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.savefig('graph4_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 4 saved!')

## Step 4: Feature Scaling (Critical Step)

In [ ]:
X = df.drop('HousePrice', axis=1)
y = df['HousePrice']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('Before Scaling:')
print(X.describe().loc[['mean', 'std', 'min', 'max']].round(3))

In [ ]:
print('After Scaling:')
print(X_scaled.describe().loc[['mean', 'std', 'min', 'max']].round(3))

In [ ]:
# --- Graph 5: Before vs After Scaling (MedInc feature) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X['MedInc'], bins=40, color='tomato', edgecolor='white')
axes[0].set_title('Before Scaling — MedInc', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Original Values')
axes[0].set_ylabel('Frequency')

axes[1].hist(X_scaled['MedInc'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('After StandardScaler — MedInc', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Scaled Values (mean=0, std=1)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Graph 5: Effect of Feature Scaling (MedInc)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('graph5_scaling_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 5 saved!')

## Step 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Testing set size  : {X_test.shape[0]} samples')
print(f'Train ratio       : {X_train.shape[0]/len(X)*100:.1f}%')
print(f'Test ratio        : {X_test.shape[0]/len(X)*100:.1f}%')

## Step 6: Train Multiple Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42)
}

print('Models to train:')
for name in models:
    print(f'  → {name}')

## Step 7: Model Evaluation and Comparison

In [ ]:
results = {}
predictions_dict = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    predictions_dict[name] = predictions

    rmse = mean_squared_error(y_test, predictions, squared=False)
    mae  = mean_absolute_error(y_test, predictions)
    r2   = r2_score(y_test, predictions)

    results[name] = {'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'R2 Score': round(r2, 4)}
    print(f'{name:25s} → RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}')

results_df = pd.DataFrame(results).T
print('\n📊 Model Comparison Table:')
results_df

In [ ]:
# Highlight best model
best_model_name = results_df['R2 Score'].idxmax()
print(f'🏆 Best Model: {best_model_name}')
print(f'   R² Score : {results_df.loc[best_model_name, "R2 Score"]}')
print(f'   RMSE     : {results_df.loc[best_model_name, "RMSE"]}')

## Step 8: Visual Performance Validation — All Graphs

In [ ]:
# --- Graph 6: Model Comparison Bar Chart (RMSE, MAE, R²) ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_names = list(results.keys())
colors = ['#2196F3', '#FF9800', '#4CAF50']

metrics = ['RMSE', 'MAE', 'R2 Score']
titles = ['RMSE (Lower = Better)', 'MAE (Lower = Better)', 'R² Score (Higher = Better)']

for i, (metric, title) in enumerate(zip(metrics, titles)):
    vals = [results[m][metric] for m in model_names]
    bars = axes[i].bar(model_names, vals, color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=10)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Graph 6: Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('graph6_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 6 saved!')

In [ ]:
# --- Graph 7: Actual vs Predicted — All 3 Models ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, preds) in enumerate(predictions_dict.items()):
    r2 = results[name]['R2 Score']
    axes[i].scatter(y_test, preds, alpha=0.3, color=colors[i], s=15)
    # Perfect prediction line
    line = [y_test.min(), y_test.max()]
    axes[i].plot(line, line, 'r--', linewidth=2, label='Perfect Prediction')
    axes[i].set_title(f'{name}\nR² = {r2}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Actual House Prices')
    axes[i].set_ylabel('Predicted House Prices')
    axes[i].legend(fontsize=9)

plt.suptitle('Graph 7: Actual vs Predicted House Prices — All Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('graph7_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 7 saved!')

In [ ]:
# --- Graph 8: Residual Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, preds) in enumerate(predictions_dict.items()):
    residuals = y_test.values - preds
    axes[i].scatter(preds, residuals, alpha=0.3, color=colors[i], s=15)
    axes[i].axhline(y=0, color='red', linestyle='--', linewidth=2)
    axes[i].set_title(f'{name} — Residuals', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Predicted Values')
    axes[i].set_ylabel('Residuals (Actual - Predicted)')

plt.suptitle('Graph 8: Residual Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('graph8_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 8 saved!')

In [ ]:
# --- Graph 9: Residual Distribution (Histogram) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, preds) in enumerate(predictions_dict.items()):
    residuals = y_test.values - preds
    axes[i].hist(residuals, bins=50, color=colors[i], edgecolor='white', alpha=0.85)
    axes[i].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Residual Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Graph 9: Residual Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('graph9_residual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 9 saved!')

In [ ]:
# --- Graph 10: Feature Importance — Decision Tree ---
dt_model = models['Decision Tree']
feat_imp = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors_imp = sns.color_palette('RdYlGn', len(feat_imp))
bars = plt.barh(feat_imp.index, feat_imp.values, color=colors_imp, edgecolor='white')

for bar, val in zip(bars, feat_imp.values):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

plt.title('Graph 10: Feature Importance — Decision Tree Regressor', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.savefig('graph10_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 10 saved!')

In [ ]:
# --- Graph 11: Linear Regression Coefficients ---
lr_model = models['Linear Regression']
coef_series = pd.Series(lr_model.coef_, index=X.columns).sort_values()

coef_colors = ['#d73027' if v < 0 else '#1a9850' for v in coef_series.values]
plt.figure(figsize=(10, 6))
bars = plt.barh(coef_series.index, coef_series.values, color=coef_colors, edgecolor='white')
plt.axvline(x=0, color='black', linewidth=1)
plt.title('Graph 11: Linear Regression Coefficients', fontsize=14, fontweight='bold')
plt.xlabel('Coefficient Value (after scaling)')
plt.tight_layout()
plt.savefig('graph11_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 11 saved!')

In [ ]:
# --- Graph 12: R² Score Comparison (Radar / Grouped Bar) ---
metrics_compare = pd.DataFrame(results).T

x = np.arange(len(metrics_compare.index))
width = 0.25
metric_cols = ['RMSE', 'MAE', 'R2 Score']
metric_colors = ['#e74c3c', '#f39c12', '#27ae60']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (col, mc) in enumerate(zip(metric_cols, metric_colors)):
    ax.bar(x + i*width, metrics_compare[col], width, label=col, color=mc, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_compare.index, fontsize=11)
ax.set_title('Graph 12: All Metrics — Side by Side Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Metric Value')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('graph12_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 12 saved!')

## Step 9: Final Model Selection & Conclusion

In [ ]:
print('=' * 55)
print('       FINAL MODEL PERFORMANCE SUMMARY')
print('=' * 55)
print(results_df.to_string())
print('=' * 55)
print(f'\n🏆 Best Model Selected : {best_model_name}')
print(f'   R² Score            : {results_df.loc[best_model_name, "R2 Score"]}')
print(f'   RMSE                : {results_df.loc[best_model_name, "RMSE"]}')
print(f'   MAE                 : {results_df.loc[best_model_name, "MAE"]}')
print()
print('📌 Justification:')
print('   → Highest R² Score → explains maximum variance in house prices')
print('   → Lowest RMSE → predictions closest to actual values')
print('   → Model generalizes well on unseen test data')

## Optional: Save Best Model using joblib

In [ ]:
import joblib

best_model_obj = models[best_model_name]
joblib.dump(best_model_obj, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f'✅ Best model ({best_model_name}) saved as best_model.pkl')
print('✅ Scaler saved as scaler.pkl')

---
## ✅ Task 2 Complete!

| Deliverable | Status |
|---|---|
| Jupyter Notebook | ✅ Done |
| Feature Scaling | ✅ Done |
| Multiple Models Trained | ✅ Done (3 models) |
| Model Comparison Table | ✅ Done |
| All Graphs (12 total) | ✅ Done |
| Best Model Saved | ✅ Done |

**Maincrafts Technology — AI/ML Internship Task 2**